# DLinear — Walmart Store Sales Forecasting

**Architecture:** DLinear (Decomposition + Linear). It decomposes each series into a
*trend* component (moving average) and a *seasonal/remainder* component, then fits a
simple linear layer to each. It is univariate (uses only past sales, no exogenous
features) but is a very strong, fast baseline for long-horizon forecasting.

**Data view:** one series per Store+Dept, weekly (Friday) frequency, long format
(`unique_id`, `ds`, `y`). Prepared by `src/features/nn_preprocessing.py`.

**Evaluation:** WMAE on the validation period (holiday weeks weighted 5x), the same
metric used for the tree models.

**MLflow structure:** experiment `DLinear_Training` with runs `DLinear_Data_Prep`,
`DLinear_Baseline`, `DLinear_Tuning`, `DLinear_Best_Pipeline`.

**Result:** baseline (input_size=52) = 2,679 WMAE, tuning (input_size=104, lr=1e-3) =
4,079 WMAE. The baseline won, so it is kept as the best pipeline.

In [ ]:
# Run from the project root so data/ and src/ resolve, whether launched
# from the repo root or from the notebooks/ folder.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from neuralforecast import NeuralForecast
from neuralforecast.models import DLinear
from neuralforecast.losses.pytorch import MAE

from src.features.nn_preprocessing import build_long_df, attach_static, temporal_split, FREQ
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

# Experiment tracking: DagsHub-hosted MLflow.
# First run opens a browser to authorize, then caches the token locally.
dagshub.init(repo_owner="GigiSichinava", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("DLinear_Training")

SPLIT_DATE = "2012-01-01"
SEED = 42
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape, "| stores:", stores.shape, "| features:", features.shape)

## Run 1 — Data preparation

Reshape into the long panel format and log the preprocessing choices (gap-fill,
frequency, split date, number of series, forecast horizon).

In [ ]:
with mlflow.start_run(run_name="DLinear_Data_Prep"):
    # Step 1: build the long panel and split by date.
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    static_df = attach_static(Y_df, stores)
    train_df, valid_df, horizon = temporal_split(Y_df, SPLIT_DATE)

    # Step 2: log the preprocessing choices.
    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", SPLIT_DATE)
    mlflow.log_param("gap_fill", "reindex_weekly_y0")
    mlflow.log_param("n_series", int(Y_df["unique_id"].nunique()))
    mlflow.log_param("horizon_weeks", int(horizon))
    mlflow.log_metric("train_rows", float(len(train_df)))
    mlflow.log_metric("valid_rows", float(len(valid_df)))

    print("series:", Y_df["unique_id"].nunique(), "| horizon (weeks):", horizon)
    print("train rows:", len(train_df), "| valid rows:", len(valid_df))

## WMAE evaluation helper

`neuralforecast` returns a forecast for each `unique_id`/`ds`. We inner-join it to
the validation rows (which carry the true `y` and the `IsHoliday` flag) and compute
WMAE with the shared metric.

In [ ]:
def evaluate_forecast(forecast_df, valid_df, model_col="DLinear"):
    # Join forecasts to the true validation values on series id and date.
    merged = forecast_df.merge(
        valid_df[["unique_id", "ds", "y", "IsHoliday"]],
        on=["unique_id", "ds"], how="inner",
    )
    # Neural models can output small negatives; sales can't be negative, so clip.
    preds = merged[model_col].clip(lower=0)
    wmae = calculate_wmae(merged["y"], preds, merged["IsHoliday"])
    return wmae, merged

## Run 2 — Baseline DLinear

`input_size=52` (one year lookback) forecasting the full validation horizon.
`scaler_type='robust'` normalizes each window so series of very different sizes are
handled fairly. On CPU this run takes a few minutes. **Result: WMAE 2,679.**

In [ ]:
with mlflow.start_run(run_name="DLinear_Baseline"):
    params = dict(h=horizon, input_size=52, max_steps=500, moving_avg_window=25,
                  scaler_type="robust", start_padding_enabled=True,
                  random_seed=SEED, alias="DLinear")
    mlflow.log_params(params)

    nf = NeuralForecast(models=[DLinear(loss=MAE(), **params)], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "DLinear")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"DLinear baseline WMAE: {wmae:,.2f}")

## Run 3 — Tuning

A second configuration: longer lookback (`input_size=104`), more steps, higher
learning rate. **Result: WMAE 4,079 — worse than the baseline.** The higher learning
rate (1e-3) likely overshot; the shorter, gentler baseline generalized better.

In [ ]:
with mlflow.start_run(run_name="DLinear_Tuning"):
    params = dict(h=horizon, input_size=104, max_steps=1000, moving_avg_window=25,
                  scaler_type="robust", learning_rate=1e-3, start_padding_enabled=True,
                  random_seed=SEED, alias="DLinear")
    mlflow.log_params(params)

    nf = NeuralForecast(models=[DLinear(loss=MAE(), **params)], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "DLinear")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"DLinear tuned WMAE: {wmae:,.2f}")

## Run 4 — Best pipeline (retrain on all data, save for inference)

The baseline config won (2,679 vs 4,079), so `BEST` is the baseline. Retrain it on the
full history (train + validation) and save the fitted model to `saved_models/` and as
an MLflow artifact. The inference notebook will load this to make the Kaggle submission.

In [9]:
# Best config = the baseline (lower WMAE).
BEST = dict(h=horizon, input_size=52, max_steps=500, moving_avg_window=25,
            scaler_type="robust", start_padding_enabled=True,
            random_seed=SEED, alias="DLinear")

with mlflow.start_run(run_name="DLinear_Best_Pipeline"):
    mlflow.log_params(BEST)

    # Retrain on the full history so the saved model uses every available week.
    nf_best = NeuralForecast(models=[DLinear(loss=MAE(), **BEST)], freq=FREQ)
    nf_best.fit(df=Y_df[["unique_id", "ds", "y"]])

    save_path = "saved_models/dlinear_nf"
    os.makedirs(save_path, exist_ok=True)
    nf_best.save(path=save_path, overwrite=True, save_dataset=False)
    mlflow.log_artifacts(save_path, artifact_path="dlinear_nf")
    print("Saved DLinear model to", save_path)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name          | Type          | Params | Mode 
--------------------------------------------------------
0 | loss          | MAE           | 0      | train
1 | padder_train  | ConstantPad1d | 0      | train
2 | scaler        | TemporalNorm  | 0      | train
3 | decomp        | SeriesDecomp  | 0      | train
4 | linear_trend  | Linear        | 2.3 K  | train
5 | linear_season | Linear        | 2.3 K  | train
--------------------------------------------------------
4.6 K     Trainable params
0         Non-trainable params
4.6 K     Total params
0.018     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode


Epoch 0:  95%|█████████▌| 100/105 [00:02<00:00, 44.49it/s, v_num=10, train_loss_step=53.80] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  90%|█████████ | 95/105 [00:01<00:00, 53.45it/s, v_num=10, train_loss_step=28.00, train_loss_epoch=104.0] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  86%|████████▌ | 90/105 [00:02<00:00, 42.44it/s, v_num=10, train_loss_step=15.60, train_loss_epoch=170.0]   
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  81%|████████  | 85/105 [00:01<00:00, 50.12it/s, v_num=10, train_loss_step=54.10, train_loss_epoch=64.10]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 80/80 [00:01<00:00, 53.69it/s, v_num=10, train_loss_step=20.30, train_loss_epoch=71.50]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0

`Trainer.fit` stopped: `max_steps=500` reached.


Epoch 4: 100%|██████████| 80/80 [00:02<00:00, 38.09it/s, v_num=10, train_loss_step=20.30, train_loss_epoch=69.20]
Saved DLinear model to saved_models/dlinear_nf
🏃 View run DLinear_Best_Pipeline at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/0/runs/01b6a51f3bf54a7495c6887359a1a5b4
🧪 View experiment at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/0


## Notes

- DLinear is univariate: it ignores the exogenous columns on purpose. The
  exogenous-feature comparison is done in the PatchTST notebook.
- Runs are logged to DagsHub — open the **Experiments** tab of your repo
  (https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting) to compare them.
- Baseline WMAE 2,679 vs tuned 4,079: kept the baseline as the best model.